# PVGIS quickstart — hourly solar radiation (live)

The `pvgis` backend fetches solar-radiation / PV time series from the keyless
[JRC PVGIS 5.3](https://re.jrc.ec.europa.eu/pvg_tools/) REST API. No
credentials, no SDK. This notebook pulls one year of **hourly** radiation for a
single point (northern Italy) and inspects the returned `pandas.DataFrame`.

In [ ]:
import tempfile

import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens

OUT = tempfile.mkdtemp()  # downloads also write a CSV here

## Download one year of `seriescalc`

`variables=["seriescalc"]` selects the hourly radiation tool. A single
`point=(lat, lon)` is one keyless GET; the `start` / `end` years bound the
window. `download()` returns the hourly table and writes it to `path` as CSV.

In [ ]:
df = EarthLens(
    data_source="pvgis",
    variables=["seriescalc"],
    start="2020-01-01",
    end="2020-12-31",
    point=(45.0, 8.0),
    path=OUT,
).download(progress_bar=False)

print(df.shape)
df.head()

Each row is one hour. `G(i)` is global in-plane irradiance (W/m²), `T2m` the
air temperature (°C), `WS10m` the wind speed; `lat` / `lon` / `product` tag the
sampled point. The `time` column is a parsed `datetime64`.

In [ ]:
df.dtypes

## Plot a few days of irradiance

A short window shows the diurnal cycle of `G(i)`.

In [ ]:
window = df[df["time"] < "2020-06-04"]
window = window[window["time"] >= "2020-06-01"]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(window["time"], window["G(i)"])
ax.set_ylabel("G(i)  [W/m²]")
ax.set_title("PVGIS seriescalc — in-plane irradiance, 1–3 June 2020 (45°N, 8°E)")
fig.autofmt_xdate()
plt.show()

## Takeaway

One keyless call returns a tidy hourly `DataFrame` you can resample or plot
directly. PVGIS already returns the resolved hourly series, so the facade
rejects `aggregate=`; use pandas (`df.set_index("time").resample("1D").mean()`)
for a coarser cadence.